# ScriptConsist: Gajendra and Airavata
Same settings as before: *GPU T4 x2*, *Internet on*, dataset **scriptconsist-code** (the **new** version), secret `HF_TOKEN`.
Run the cells in order. Send me the output of the two check cells before relying on the full runs.

In [ ]:
import os, shutil, glob
WORK = '/kaggle/working/scriptconsist'
hits = glob.glob('/kaggle/input/**/code/run_generation.py', recursive=True)
assert hits, 'Dataset not attached'
SRC = os.path.dirname(os.path.dirname(hits[0]))
os.makedirs(WORK, exist_ok=True)
for sub in ('code', 'data'):   # copy code and data only; never touch results/
    shutil.copytree(os.path.join(SRC, sub), os.path.join(WORK, sub), dirs_exist_ok=True)
os.chdir(WORK)
assert '--prompt-format' in open('code/run_generation.py').read(), 'Old code version attached - update the dataset to the new zip'
print('results so far:', os.listdir('results/generations') if os.path.exists('results/generations') else 'none')

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes rapidfuzz aksharamukha sentencepiece protobuf
import torch; print(torch.cuda.device_count(), 'GPUs')

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret('HF_TOKEN'))

## Gajendra (BhabhaAI)

In [ ]:
!python code/run_generation.py --model BhabhaAI/Gajendra-v0.1 --show-template

In [ ]:
# 10-question test: uses the built-in chat format if the model has one, otherwise the tulu format
import subprocess, json
out = subprocess.run(['python','code/run_generation.py','--model','BhabhaAI/Gajendra-v0.1','--show-template'],capture_output=True,text=True).stdout
FMT_G = 'chat' if 'chat template present: True' in out else 'chatml'   # Gajendra was trained on ChatML
print('Gajendra prompt format:', FMT_G)
!python code/run_generation.py --model BhabhaAI/Gajendra-v0.1 --precision fp16 --prompt-format {FMT_G} --limit 10
try:
    for l in list(open('results/generations/BhabhaAI__Gajendra-v0.1__fp16__limit10.jsonl'))[:15]:
        r = json.loads(l); print(r['form'], '|', r['response'][:110])
except FileNotFoundError:
    print('No output - send me the messages above')

In [ ]:
!python code/run_generation.py --model BhabhaAI/Gajendra-v0.1 --precision fp16 --prompt-format {FMT_G}
!rm -rf ~/.cache/huggingface/hub

## Airavata (AI4Bharat)

In [ ]:
!python code/run_generation.py --model ai4bharat/Airavata --show-template --prompt-format tulu

In [ ]:
!python code/run_generation.py --model ai4bharat/Airavata --precision fp16 --prompt-format tulu --limit 10
try:
    for l in list(open('results/generations/ai4bharat__Airavata__fp16__limit10.jsonl'))[:15]:
        r = json.loads(l); print(r['form'], '|', r['response'][:110])
except FileNotFoundError:
    print('No output - send me the messages above')

In [ ]:
!python code/run_generation.py --model ai4bharat/Airavata --precision fp16 --prompt-format tulu
!rm -rf ~/.cache/huggingface/hub

## Package

In [ ]:
full=[p for p in glob.glob('results/generations/*.jsonl') if 'limit' not in p]
!python code/score.py {' '.join(full)}
shutil.make_archive('/kaggle/working/results_indic', 'zip', 'results')
print('Download results_indic.zip from the Output panel')